# Phase 2: Data Preprocessing & Feature Engineering

**Date:** March 13, 2026  
**Dataset:** 17,547 households × 365 days (2023)  
**Objective:** Clean data, extract features, prepare for clustering

---

## Table of Contents

1. [Setup & Imports](#1-Setup-&-Imports)
2. [Load Data](#2-Load-Data)
3. [Data Cleaning Pipeline](#3-Data-Cleaning-Pipeline)
4. [Feature Extraction](#4-Feature-Extraction)
5. [Feature Analysis](#5-Feature-Analysis)
6. [Feature Selection](#6-Feature-Selection)
7. [Dimensionality Reduction](#7-Dimensionality-Reduction)
8. [Data Scaling & Export](#8-Data-Scaling-&-Export)
9. [Summary & Next Steps](#9-Summary-&-Next-Steps)

---

## 1. Setup & Imports

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Reload modules to pick up latest changes
import importlib
if 'src.utils.data_loader' in sys.modules:
    import src.utils.data_loader
    importlib.reload(src.utils.data_loader)

# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Our preprocessing modules
from src.preprocessing.data_cleaning import DataCleaner
from src.preprocessing.feature_engineering import FeatureEngineer
from src.preprocessing.feature_selection import FeatureSelector

# Utility functions
from src.utils.data_loader import load_sample_data
from src.utils.config import DATA_DIR, RESULTS_DIR

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ All imports successful!")
print(f"📁 Data directory: {DATA_DIR}")
print(f"📁 Results directory: {RESULTS_DIR}")
print(f"⏰ Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

ImportError: cannot import name 'load_sample_data' from 'src.utils.data_loader' (c:\Users\Lundrim\Desktop\RDK Projectt\src\utils\data_loader.py)

## 2. Load Data

Load the 2023 training data from Phase 1.

In [ ]:
# Load 2023 data
print("📥 Loading sample_23.csv...")
df_23 = load_sample_data(year=23)

print(f"\n📊 Data Summary:")
print(f"   - Shape: {df_23.shape}")
print(f"   - Households: {df_23.shape[0]:,}")
print(f"   - Days: {df_23.shape[1]:,}")
print(f"   - Memory usage: {df_23.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display first few rows
print("\n📋 First 5 households (first 10 days):")
df_23.iloc[:5, :10]

📥 Loading sample_23.csv...


NameError: name 'load_sample_data' is not defined

## 3. Data Cleaning Pipeline

Apply comprehensive data cleaning:
- Handle missing values (if any)
- Handle outliers (IQR method with capping)
- Handle zeros (keep as valid)
- Handle negatives (if any)

In [ ]:
# Initialize DataCleaner
cleaner = DataCleaner()

# Check for missing values
missing_count = df_23.isna().sum().sum()
print(f"🔍 Missing values: {missing_count}")

# Check for negative values
negative_count = (df_23 < 0).sum().sum()
print(f"🔍 Negative values: {negative_count}")

# Check for zero values
zero_count = (df_23 == 0).sum().sum()
zero_pct = (zero_count / df_23.size) * 100
print(f"🔍 Zero values: {zero_count:,} ({zero_pct:.2f}% of all data points)")

print("\n🧹 Applying cleaning pipeline...")
df_cleaned = cleaner.clean_pipeline(
    df_23,
    handle_missing=True,
    handle_outliers=True,
    handle_zeros=False,  # Keep zeros as valid
    handle_negatives=True,
    normalize=False  # Will normalize after feature extraction
)

print("\n✅ Cleaning complete!")
print(f"   - Original shape: {df_23.shape}")
print(f"   - Cleaned shape: {df_cleaned.shape}")
print(f"   - Households preserved: {df_cleaned.shape[0]:,}")

### 3.1 Before vs After Cleaning Comparison

In [ ]:
# Compare distributions before and after cleaning
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Before cleaning
all_values_before = df_23.values.flatten()
axes[0].hist(all_values_before[all_values_before > 0], bins=100, edgecolor='black', alpha=0.7)
axes[0].set_title('Before Cleaning (excluding zeros)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Consumption (kWh/day)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(np.mean(all_values_before[all_values_before > 0]), 
                color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(all_values_before[all_values_before > 0]):.2f}')
axes[0].legend()

# After cleaning
all_values_after = df_cleaned.values.flatten()
axes[1].hist(all_values_after[all_values_after > 0], bins=100, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title('After Cleaning (excluding zeros)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Consumption (kWh/day)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(np.mean(all_values_after[all_values_after > 0]), 
                color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(all_values_after[all_values_after > 0]):.2f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'cleaning_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\n📊 Summary Statistics:")
print(f"\nBefore Cleaning:")
print(f"   - Mean: {np.mean(all_values_before):.3f} kWh/day")
print(f"   - Median: {np.median(all_values_before):.3f} kWh/day")
print(f"   - Std: {np.std(all_values_before):.3f} kWh/day")
print(f"   - Max: {np.max(all_values_before):.3f} kWh/day")

print(f"\nAfter Cleaning:")
print(f"   - Mean: {np.mean(all_values_after):.3f} kWh/day")
print(f"   - Median: {np.median(all_values_after):.3f} kWh/day")
print(f"   - Std: {np.std(all_values_after):.3f} kWh/day")
print(f"   - Max: {np.max(all_values_after):.3f} kWh/day")

## 4. Feature Extraction

Extract comprehensive features (69+) across 5 categories:
1. **Statistical** (16 features): mean, median, std, quantiles, moments
2. **Temporal** (25+ features): day-of-week, monthly, quarterly patterns
3. **Seasonality** (8 features): seasonal means, ratios, amplitude
4. **Variability** (13 features): rolling stats, peaks, autocorrelation
5. **Shape** (7 features): entropy, FFT, Hurst exponent

In [ ]:
# Initialize FeatureEngineer
engineer = FeatureEngineer()

print("🔧 Extracting features from 17,547 households...")
print("⏱️ This may take several minutes...\n")

# Extract all features
df_features = engineer.extract_all_features(
    df_cleaned,
    statistical=True,
    temporal=True,
    seasonality=True,
    variability=True,
    shape=True
)

print(f"\n✅ Feature extraction complete!")
print(f"   - Shape: {df_features.shape}")
print(f"   - Households: {df_features.shape[0]:,}")
print(f"   - Features extracted: {df_features.shape[1]:,}")
print(f"   - Memory usage: {df_features.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display first few features
print("\n📋 First 10 features:")
print(df_features.columns[:10].tolist())

# Display feature summary
print("\n📊 Feature Summary:")
df_features.head()

### 4.1 Feature Category Breakdown

In [ ]:
# Categorize features
statistical_features = [col for col in df_features.columns if any(x in col for x in ['mean', 'median', 'std', 'min', 'max', 'q25', 'q50', 'q75', 'iqr', 'skewness', 'kurtosis', 'cv', 'zero', 'variance', 'range'])]
temporal_features = [col for col in df_features.columns if any(x in col for x in ['dow_', 'weekday', 'weekend', 'month_', 'quarter_', 'trend'])]
seasonality_features = [col for col in df_features.columns if any(x in col for x in ['winter', 'spring', 'summer', 'fall', 'seasonal'])]
variability_features = [col for col in df_features.columns if any(x in col for x in ['rolling', 'peak', 'valley', 'autocorr', 'diff'])]
shape_features = [col for col in df_features.columns if any(x in col for x in ['entropy', 'spectral', 'hurst', 'linearity'])]

print("📊 Feature Breakdown by Category:\n")
print(f"1. Statistical Features ({len(statistical_features)}):")
print(f"   {statistical_features[:8]}...\n")

print(f"2. Temporal Features ({len(temporal_features)}):")
print(f"   {temporal_features[:8]}...\n")

print(f"3. Seasonality Features ({len(seasonality_features)}):")
print(f"   {seasonality_features}\n")

print(f"4. Variability Features ({len(variability_features)}):")
print(f"   {variability_features}\n")

print(f"5. Shape Features ({len(shape_features)}):")
print(f"   {shape_features}\n")

print(f"\n📈 Total Features: {len(df_features.columns)}")

### 4.2 Save All Features

In [ ]:
# Save all extracted features
features_all_path = os.path.join(RESULTS_DIR, 'features_all.csv')
df_features.to_csv(features_all_path)
print(f"💾 Saved all features to: {features_all_path}")
print(f"   - Size: {os.path.getsize(features_all_path) / 1024**2:.2f} MB")

## 5. Feature Analysis

Analyze feature distributions and relationships.

In [ ]:
# Basic statistics
print("📊 Feature Statistics:\n")
df_features.describe()

### 5.1 Feature Distributions

In [ ]:
# Plot distributions of key features
key_features = ['mean', 'std', 'cv', 'skewness', 'kurtosis', 'trend_slope', 'seasonal_amplitude', 'entropy']
key_features = [f for f in key_features if f in df_features.columns]

if len(key_features) > 0:
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.flatten()

    for i, feature in enumerate(key_features):
        if i < 8:
            axes[i].hist(df_features[feature].dropna(), bins=50, edgecolor='black', alpha=0.7)
            axes[i].set_title(feature, fontsize=10, fontweight='bold')
            axes[i].set_xlabel('Value')
            axes[i].set_ylabel('Frequency')
            axes[i].axvline(df_features[feature].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
            axes[i].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'feature_distributions.png'), dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("⚠️ No key features found for plotting")

### 5.2 Feature Correlation Analysis

In [ ]:
# Initialize FeatureSelector
selector = FeatureSelector()

# Calculate correlations
print("🔍 Calculating feature correlations...")
corr_matrix = selector.calculate_correlations(df_features)

# Plot correlation matrix (sample of features)
sample_features = df_features.columns[:20]  # First 20 features for visualization
corr_sample = corr_matrix.loc[sample_features, sample_features]

plt.figure(figsize=(14, 12))
sns.heatmap(corr_sample, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix (First 20 Features)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'correlation_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

# Find highly correlated pairs
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr_pairs.append((
                corr_matrix.columns[i], 
                corr_matrix.columns[j], 
                corr_matrix.iloc[i, j]
            ))

print(f"\n📊 Highly Correlated Feature Pairs (|r| > 0.9): {len(high_corr_pairs)}")
if len(high_corr_pairs) > 0:
    print("\nTop 10 pairs:")
    for feat1, feat2, corr in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True)[:10]:
        print(f"   - {feat1} ↔ {feat2}: {corr:.3f}")

## 6. Feature Selection

Remove highly correlated features (threshold = 0.9) and low-variance features.

In [ ]:
# Remove highly correlated features
print("🔧 Removing highly correlated features (threshold=0.9)...")
df_selected, removed_features = selector.remove_highly_correlated(df_features, threshold=0.9)

print(f"\n✅ Feature selection complete!")
print(f"   - Original features: {df_features.shape[1]}")
print(f"   - Selected features: {df_selected.shape[1]}")
print(f"   - Removed features: {len(removed_features)}")
print(f"   - Reduction: {(1 - df_selected.shape[1] / df_features.shape[1]) * 100:.1f}%")

if len(removed_features) > 0:
    print(f"\n📋 Removed features (sample):")
    print(f"   {removed_features[:10]}")

### 6.1 Variance-Based Selection

In [ ]:
# Check for low-variance features
print("🔍 Checking for low-variance features (threshold=0.01)...")
df_variance_selected = selector.select_by_variance(df_selected, threshold=0.01)

print(f"\n✅ Variance filtering complete!")
print(f"   - Before variance filter: {df_selected.shape[1]}")
print(f"   - After variance filter: {df_variance_selected.shape[1]}")
print(f"   - Removed: {df_selected.shape[1] - df_variance_selected.shape[1]}")

# Use variance-filtered features for next steps
df_selected = df_variance_selected

### 6.2 Save Selected Features

In [ ]:
# Save selected features
features_selected_path = os.path.join(RESULTS_DIR, 'features_selected.csv')
df_selected.to_csv(features_selected_path)
print(f"💾 Saved selected features to: {features_selected_path}")
print(f"   - Size: {os.path.getsize(features_selected_path) / 1024**2:.2f} MB")

## 7. Dimensionality Reduction

Apply PCA, t-SNE, and UMAP for dimensionality reduction and visualization.

### 7.1 PCA (Principal Component Analysis)

In [ ]:
# Apply PCA with auto-component selection (95% variance)
print("🔧 Applying PCA (95% variance threshold)...")
df_pca, pca_model = selector.apply_pca(df_selected, n_components=None, variance_threshold=0.95)

print(f"\n✅ PCA complete!")
print(f"   - Original features: {df_selected.shape[1]}")
print(f"   - Principal components: {df_pca.shape[1]}")
print(f"   - Variance explained: {pca_model.explained_variance_ratio_.sum():.3f}")
print(f"   - Reduction: {(1 - df_pca.shape[1] / df_selected.shape[1]) * 100:.1f}%")

# Plot variance explained
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(pca_model.explained_variance_ratio_) + 1), 
         pca_model.explained_variance_ratio_, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Principal Component', fontsize=12)
plt.ylabel('Variance Explained', fontsize=12)
plt.title('Variance Explained by Each PC', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
cumsum = np.cumsum(pca_model.explained_variance_ratio_)
plt.plot(range(1, len(cumsum) + 1), cumsum, 'ro-', linewidth=2, markersize=8)
plt.axhline(y=0.95, color='green', linestyle='--', linewidth=2, label='95% Threshold')
plt.xlabel('Number of Components', fontsize=12)
plt.ylabel('Cumulative Variance Explained', fontsize=12)
plt.title('Cumulative Variance Explained', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'pca_variance_explained.png'), dpi=300, bbox_inches='tight')
plt.show()

### 7.2 Feature Importance from PCA

In [ ]:
# Get feature importance from PCA loadings
feature_importance = selector.get_feature_importance_pca(df_selected, n_components=5)

print("📊 Top 15 Most Important Features (based on PCA loadings):\n")
print(feature_importance.head(15))

# Plot top features
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(20)
plt.barh(range(len(top_features)), top_features['importance'], color='steelblue', edgecolor='black')
plt.yticks(range(len(top_features)), top_features['feature'], fontsize=10)
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 20 Features by PCA Importance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'pca_feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()

### 7.3 Save PCA Features

In [ ]:
# Save PCA features
pca_features_path = os.path.join(RESULTS_DIR, 'pca_features.csv')
df_pca.to_csv(pca_features_path)
print(f"💾 Saved PCA features to: {pca_features_path}")
print(f"   - Size: {os.path.getsize(pca_features_path) / 1024**2:.2f} MB")

### 7.4 t-SNE Visualization

In [ ]:
# Apply t-SNE (on sample for speed)
print("🔧 Applying t-SNE for 2D visualization...")
print("⏱️ Using sample of 5,000 households for speed...\n")

# Sample for t-SNE (computationally expensive)
sample_size = min(5000, len(df_selected))
df_sample = df_selected.sample(n=sample_size, random_state=42)

df_tsne = selector.apply_tsne(df_sample, n_components=2, perplexity=30.0, random_state=42)

print(f"✅ t-SNE complete!")
print(f"   - Sample size: {sample_size:,}")
print(f"   - Output shape: {df_tsne.shape}")

# Plot t-SNE
plt.figure(figsize=(10, 8))
plt.scatter(df_tsne['tsne_1'], df_tsne['tsne_2'], 
            alpha=0.5, s=20, c=df_sample['mean'] if 'mean' in df_sample.columns else 'steelblue',
            cmap='viridis', edgecolors='none')
plt.xlabel('t-SNE Component 1', fontsize=12)
plt.ylabel('t-SNE Component 2', fontsize=12)
plt.title(f't-SNE Visualization ({sample_size:,} households)', fontsize=14, fontweight='bold')
if 'mean' in df_sample.columns:
    plt.colorbar(label='Mean Consumption (kWh/day)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'tsne_visualization.png'), dpi=300, bbox_inches='tight')
plt.show()

# Save t-SNE features (full dataset - may take longer)
print("\n🔧 Computing t-SNE for full dataset...")
print("⏱️ This may take several minutes...\n")
df_tsne_full = selector.apply_tsne(df_selected, n_components=2, perplexity=30.0, random_state=42)
tsne_features_path = os.path.join(RESULTS_DIR, 'tsne_features.csv')
df_tsne_full.to_csv(tsne_features_path)
print(f"💾 Saved t-SNE features to: {tsne_features_path}")

### 7.5 UMAP Visualization (Optional)

In [ ]:
# Apply UMAP if available
try:
    print("🔧 Applying UMAP for 2D visualization...")
    print("⏱️ Using sample of 5,000 households for speed...\n")
    
    df_umap = selector.apply_umap(df_sample, n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    
    print(f"✅ UMAP complete!")
    print(f"   - Output shape: {df_umap.shape}")
    
    # Plot UMAP
    plt.figure(figsize=(10, 8))
    plt.scatter(df_umap['umap_1'], df_umap['umap_2'], 
                alpha=0.5, s=20, c=df_sample['mean'] if 'mean' in df_sample.columns else 'coral',
                cmap='viridis', edgecolors='none')
    plt.xlabel('UMAP Component 1', fontsize=12)
    plt.ylabel('UMAP Component 2', fontsize=12)
    plt.title(f'UMAP Visualization ({sample_size:,} households)', fontsize=14, fontweight='bold')
    if 'mean' in df_sample.columns:
        plt.colorbar(label='Mean Consumption (kWh/day)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'umap_visualization.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
except Exception as e:
    print(f"⚠️ UMAP not available or failed: {e}")
    print("   Install with: pip install umap-learn")

## 8. Data Scaling & Export

Scale features using StandardScaler and save for clustering.

In [ ]:
# Scale selected features
print("🔧 Scaling features (StandardScaler)...")
df_scaled, scaler = selector.scale_features(df_selected, method='standard')

print(f"\n✅ Scaling complete!")
print(f"   - Shape: {df_scaled.shape}")
print(f"   - Mean: {df_scaled.mean().mean():.6f}")
print(f"   - Std: {df_scaled.std().mean():.6f}")

# Verify scaling
print("\n📊 Scaled Feature Statistics (first 5 features):")
print(df_scaled.iloc[:, :5].describe())

### 8.1 Save Scaled Features

In [ ]:
# Save scaled features
features_scaled_path = os.path.join(RESULTS_DIR, 'features_scaled.csv')
df_scaled.to_csv(features_scaled_path)
print(f"💾 Saved scaled features to: {features_scaled_path}")
print(f"   - Size: {os.path.getsize(features_scaled_path) / 1024**2:.2f} MB")

### 8.2 Save Scaler Object

In [ ]:
# Save scaler for future use
import pickle

scaler_path = os.path.join(RESULTS_DIR, 'scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f"💾 Saved scaler to: {scaler_path}")
print(f"   - Size: {os.path.getsize(scaler_path) / 1024:.2f} KB")
print("\n✅ Scaler can be loaded later for consistent transformations!")

## 9. Summary & Next Steps

In [ ]:
print("="*80)
print("   Phase 2: Data Preprocessing & Feature Engineering - COMPLETE ✅")
print("="*80)

print("\n📊 Summary:\n")

print(f"1. Data Cleaning:")
print(f"   - Outliers handled with IQR capping")
print(f"   - Zeros preserved ({zero_pct:.2f}% of data)")
print(f"   - All {df_cleaned.shape[0]:,} households preserved")

print(f"\n2. Feature Extraction:")
print(f"   - Total features extracted: {df_features.shape[1]}")
print(f"   - Statistical: {len(statistical_features)}")
print(f"   - Temporal: {len(temporal_features)}")
print(f"   - Seasonality: {len(seasonality_features)}")
print(f"   - Variability: {len(variability_features)}")
print(f"   - Shape: {len(shape_features)}")

print(f"\n3. Feature Selection:")
print(f"   - Original features: {df_features.shape[1]}")
print(f"   - After correlation filter: {df_selected.shape[1]}")
print(f"   - Reduction: {(1 - df_selected.shape[1] / df_features.shape[1]) * 100:.1f}%")

print(f"\n4. Dimensionality Reduction:")
print(f"   - PCA components: {df_pca.shape[1]}")
print(f"   - Variance explained: {pca_model.explained_variance_ratio_.sum():.1f}%")
print(f"   - t-SNE: 2D projection computed")

print(f"\n5. Data Scaling:")
print(f"   - Method: StandardScaler (z-score)")
print(f"   - Features scaled: {df_scaled.shape[1]}")
print(f"   - Scaler saved for reproducibility")

print("\n📁 Generated Files:\n")
files_generated = [
    features_all_path,
    features_selected_path,
    features_scaled_path,
    pca_features_path,
    tsne_features_path,
    scaler_path
]

for filepath in files_generated:
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / 1024**2
        print(f"   ✅ {os.path.basename(filepath)} ({size_mb:.2f} MB)")

print("\n🎯 Next Steps (Phase 3: Clustering):\n")
print("   1. Load features_scaled.csv")
print("   2. Apply K-Means, DBSCAN, Hierarchical clustering")
print("   3. Determine optimal number of clusters")
print("   4. Evaluate clustering quality (silhouette, Davies-Bouldin)")
print("   5. Visualize clusters using t-SNE/UMAP")
print("   6. Interpret cluster characteristics")

print("\n⏰ Notebook completed:", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print("="*80)

---

## Phase 2 Complete! ✅

**Key Achievements:**
- ✅ Comprehensive data cleaning pipeline
- ✅ 69+ features extracted across 5 categories
- ✅ Feature selection and correlation filtering
- ✅ PCA dimensionality reduction (95% variance)
- ✅ t-SNE/UMAP visualizations
- ✅ Feature scaling and persistence

**Ready for Phase 3: Clustering!** 🚀

---